In [1]:
import pandas as pd
import numpy as np

path = "/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/Сообщения Vertical (5).xlsx"
df = pd.read_excel(path)

df["date_add"] = pd.to_datetime(df["date_add"])
df["is_admin"] = df["from_admin"].fillna(0).astype(int)

# Focus MVP on guest messages only
guest = df[df["is_admin"] == 0].copy()

# Basic hygiene
guest["message"] = guest["message"].astype(str).str.strip()
guest = guest[guest["message"].str.len() > 0]

# Booking-level aggregation scaffold (you’ll join sentiment labels later)
booking = guest.groupby(["hotel_id", "ID_booking"]).agg(
    n_guest_msgs=("message", "size"),
    first_msg=("date_add", "min"),
    last_msg=("date_add", "max"),
).reset_index()

booking["duration_hr"] = (booking["last_msg"] - booking["first_msg"]).dt.total_seconds() / 3600
booking.head()


,hotel_id,ID_booking,n_guest_msgs,first_msg,last_msg,duration_hr
0,1,29204,37,2023-10-05 14:23:16,2024-09-11 14:41:12,8208.298889
1,1,55489,4,2023-09-21 15:11:36,2023-09-24 15:03:24,71.863333
2,1,68575,7,2023-12-21 00:08:05,2024-01-15 00:44:24,600.605278
3,1,68934,2,2023-10-17 15:30:15,2023-10-17 16:11:34,0.688611
4,1,69417,8,2023-09-19 10:25:43,2023-09-30 20:30:47,274.084444


In [2]:
guest = guest.sort_values(
    ["hotel_id", "ID_booking", "date_add"]
).reset_index(drop=True)

In [3]:
# Thread detection by time gap within a booking
GAP_HOURS = 24

df_sorted = df.sort_values(["ID_booking", "date_add"])
df_sorted["prev_time"] = df_sorted.groupby("ID_booking")["date_add"].shift(1)
df_sorted["gap_hrs"] = (df_sorted["date_add"] - df_sorted["prev_time"]).dt.total_seconds() / 3600

# New thread = first message OR gap > threshold
df_sorted["new_thread"] = (
    df_sorted["prev_time"].isna() |
    (df_sorted["gap_hrs"] > GAP_HOURS)
)
df_sorted["thread_id"] = df_sorted.groupby("ID_booking")["new_thread"].cumsum()

# Now each (ID_booking, thread_id) is one conversation thread


In [4]:
import os, json
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

"""
You classify hotel guest conversation threads into exactly one category.

PROBLEM: The guest needs physical action — someone must come to the room or
  perform a task. Includes: maintenance, cleaning requests, broken items,
  wifi voucher delivery, key issues, noise from neighbors requiring staff.

QUESTION: The guest needs information only — a text reply fully resolves it.
  Includes: checkout time, hotel policies, directions, amenity locations,
  restaurant hours, parking info, wifi password (if just the password, not a voucher).

OTHER: Does not fit above. You MUST provide a brief reason in the 'reason' field.
  Examples: pure greeting, payment/cancellation dispute, review threat.

Return ONLY JSON:
{"category": "PROBLEM"|"QUESTION"|"OTHER", "reason": "...", "confidence": 0..1}
"""

def classify_thread(thread_messages: list[str]) -> dict:
    combined = "\n---\n".join(thread_messages)  # join all messages in thread
    # ... API call

In [5]:
BATCH_SIZE = 25

texts = guest["message"].astype(str).tolist()
sent, conf = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    out = classify_batch(batch)
    sent.extend([x["sentiment"] for x in out])
    conf.extend([x.get("confidence") for x in out])
    if i % (BATCH_SIZE*50) == 0 and i > 0:
        pd.DataFrame({"sentiment": sent, "sentiment_score": conf}).to_parquet(
        "openai_partial.parquet", index=False
    )


guest["sentiment"] = sent
guest["sentiment_score"] = conf


  0%|          | 0/2257 [00:00<?, ?it/s]

NameError: name 'classify_batch' is not defined

In [6]:
#
# STEP 1: SENTIMENT ANALYSIS (you already have this)
# Based on: Le Wagon Deep Learning → Transformers
#

# Assuming guest["sentiment"] and guest["sentiment_score"] already exist
# If not, run your HuggingFace model code here

# Quick check
print("=== Sentiment Distribution (Guest Messages) ===")
print(guest["sentiment"].value_counts())
print()
print((guest["sentiment"].value_counts(normalize=True) * 100).round(1))

=== Sentiment Distribution (Guest Messages) ===


KeyError: 'sentiment'

In [7]:
# STEP 1: SENTIMENT ANALYSIS (you already have this)
# Based on: Le Wagon Deep Learning → Transformers
#

# Assuming guest["sentiment"] and guest["sentiment_score"] already exist
# If not, run your HuggingFace model code here
# Quick check
print("=== Sentiment Distribution (Guest Messages) ===")
print(guest["sentiment"].value_counts())
print()
print((guest["sentiment"].value_counts(normalize=True) * 100).round(1))

=== Sentiment Distribution (Guest Messages) ===


KeyError: 'sentiment'

In [ ]:
# STEP 2: MESSAGE TYPE CLASSIFICATION
# Based on: Le Wagon NLP → Keyword extraction


# Keywords for classification
COMPLAINT_KEYWORDS = [
    "возмущен", "безобразие", "хамство", "хамят",
    "кошмар", "ужас", "отвратительн", "недопустим",
    "обман", "наглость", "хватит", "достали", "надоело"
]

THREAT_KEYWORDS = [
    r"\суд\bb", "роспотребнадзор", "прокуратур", "адвокат",
    "напишу отзыв", "оставлю отзыв", "верну деньги"
]

PROBLEM_KEYWORDS = [
    "не работает", "сломан", "не включается", "течет",
    "воняет", "грязн", "холодно", "жарко", "шум"
]

import re

def classify_message(row) -> str:
    """Classify message using keywords + sentiment."""
    text = str(row["message"]).lower()
    sentiment = row["sentiment"]

    # THREAT - highest priority
    for pattern in THREAT_KEYWORDS:
        if re.search(pattern, text):
            return "THREAT"

    # COMPLAINT - strong negative language
    for kw in COMPLAINT_KEYWORDS:
        if kw in text:
            return "COMPLAINT"

    # COMPLAINT - NEG sentiment + frustration words
    if sentiment == "NEG" and any(w in text for w in ["снова", "опять", "уже", "сколько раз"]):
        return "COMPLAINT"

    # PROBLEM_REPORT
    for kw in PROBLEM_KEYWORDS:
        if kw in text:
            return "PROBLEM"

    # POSITIVE
    if sentiment == "POS" or any(w in text for w in ["спасибо", "благодар", "отлично"]):
        return "POSITIVE"

    return "NEUTRAL"

guest["msg_type"] = guest.apply(classify_message, axis=1)

print("=== Message Type Distribution ===")
print(guest["msg_type"].value_counts())

=== Message Type Distribution ===
msg_type
NEUTRAL      43381
POSITIVE      9884
PROBLEM       1732
COMPLAINT     1410
THREAT           8
Name: count, dtype: int64


In [ ]:
# STEP 3: BOOKING-LEVEL AGGREGATION
# Based on: Le Wagon Data Toolkit → GroupBy

# Sort by time
guest_sorted = guest.sort_values(["hotel_id", "ID_booking", "date_add"])

# Aggregate per booking
booking_stats = guest_sorted.groupby(["hotel_id", "ID_booking"]).agg(
    n_guest_msgs=("message", "size"),
    first_msg=("date_add", "min"),
    last_msg=("date_add", "max"),

    # Sentiment counts
    neg_count=("sentiment", lambda x: (x == "NEG").sum()),
    pos_count=("sentiment", lambda x: (x == "POS").sum()),

    # Message type counts
    complaint_count=("msg_type", lambda x: (x == "COMPLAINT").sum()),
    threat_count=("msg_type", lambda x: (x == "THREAT").sum()),
    problem_count=("msg_type", lambda x: (x == "PROBLEM").sum()),

    # Last message info
    last_sentiment=("sentiment", "last"),
    last_msg_type=("msg_type", "last")
).reset_index()

# Calculate percentages
booking_stats["neg_share"] = (booking_stats["neg_count"] / booking_stats["n_guest_msgs"] * 100).round(1)

# Conversation duration
booking_stats["duration_hours"] = (
    (booking_stats["last_msg"] - booking_stats["first_msg"]).dt.total_seconds() / 3600
).round(1)

booking_stats.head()

,hotel_id,ID_booking,n_guest_msgs,first_msg,last_msg,neg_count,pos_count,complaint_count,threat_count,problem_count,last_sentiment,last_msg_type,neg_share,duration_hours
0,1,29204,37,2023-10-05 14:23:16,2024-09-11 14:41:12,1,1,0,0,0,NEU,NEUTRAL,2.7,8208.3
1,1,55489,4,2023-09-21 15:11:36,2023-09-24 15:03:24,2,0,1,0,0,NEU,NEUTRAL,50.0,71.9
2,1,68575,7,2023-12-21 00:08:05,2024-01-15 00:44:24,1,3,0,0,1,NEU,NEUTRAL,14.3,600.6
3,1,68934,2,2023-10-17 15:30:15,2023-10-17 16:11:34,0,1,0,0,0,POS,POSITIVE,0.0,0.7
4,1,69417,8,2023-09-19 10:25:43,2023-09-30 20:30:47,1,1,1,0,0,NEU,NEUTRAL,12.5,274.1


In [ ]:
# Based on: Le Wagon Data Toolkit → Merge operations


# First guest message per booking
first_guest = (
    guest_sorted.groupby(["hotel_id", "ID_booking"])["date_add"]
    .min()
    .reset_index()
    .rename(columns={"date_add": "first_guest_time"})
)

# First admin message per booking
first_admin = (
    admin.groupby(["hotel_id", "ID_booking"])["date_add"]
    .min()
    .reset_index()
    .rename(columns={"date_add": "first_admin_time"})
)

# Admin message count per booking
admin_counts = (
    admin.groupby(["hotel_id", "ID_booking"])
    .size()
    .reset_index(name="n_admin_msgs")
)

# Merge all
booking_stats = booking_stats.merge(first_guest, on=["hotel_id", "ID_booking"], how="left")
booking_stats = booking_stats.merge(first_admin, on=["hotel_id", "ID_booking"], how="left")
booking_stats = booking_stats.merge(admin_counts, on=["hotel_id", "ID_booking"], how="left")

# Calculate response time (minutes)
booking_stats["reply_time_min"] = (
    (booking_stats["first_admin_time"] - booking_stats["first_guest_time"])
    .dt.total_seconds() / 60
).round(1)

# Handle edge cases
booking_stats["n_admin_msgs"] = booking_stats["n_admin_msgs"].fillna(0).astype(int)
booking_stats.loc[booking_stats["reply_time_min"] < 0, "reply_time_min"] = np.nan  # admin wrote first

# Admin responded? (boolean)
booking_stats["admin_responded"] = booking_stats["n_admin_msgs"] > 0

# Admin-to-guest ratio
booking_stats["admin_guest_ratio"] = (
    booking_stats["n_admin_msgs"] / booking_stats["n_guest_msgs"]
).round(2)

print("=== Admin Response Stats ===")
print(f"Bookings with admin response: {booking_stats['admin_responded'].sum()} / {len(booking_stats)}")
print(f"Average response time: {booking_stats['reply_time_min'].mean():.1f} min")
print(f"Median response time: {booking_stats['reply_time_min'].median():.1f} min")

=== Admin Response Stats ===
Bookings with admin response: 6695 / 7982
Average response time: 878.2 min
Median response time: 7.8 min


=== Risk Distribution ===
risk_level
LOW       7464
MEDIUM     419
HIGH        99
Name: count, dtype: int64


In [ ]:

# STEP 8: YEAR-OVER-YEAR TRENDS
# Based on: Le Wagon Data Toolkit → Time series grouping

guest["year"] = guest["date_add"].dt.year
guest["month"] = guest["date_add"].dt.to_period("M")

# Sentiment by hotel + year
sentiment_trend = (
    guest.groupby(["hotel_id", "year"])["sentiment"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure columns exist
for col in ["NEG", "NEU", "POS"]:
    if col not in sentiment_trend.columns:
        sentiment_trend[col] = 0.0

# Add message counts
msg_counts = guest.groupby(["hotel_id", "year"]).size().reset_index(name="n_messages")
sentiment_trend = sentiment_trend.merge(msg_counts, on=["hotel_id", "year"])

# Format as percentages
sentiment_trend[["NEG", "NEU", "POS"]] = (sentiment_trend[["NEG", "NEU", "POS"]] * 100).round(1)

print("=== Sentiment % by Hotel + Year ===")
sentiment_trend[["hotel_id", "year", "n_messages", "NEG", "NEU", "POS"]]

=== Sentiment % by Hotel + Year ===


,hotel_id,year,n_messages,NEG,NEU,POS
0,1,2023,3509,20.5,62.5,17.0
1,1,2024,6172,21.6,64.6,13.7
2,1,2025,2524,20.6,64.1,15.3
3,2,2023,555,15.5,71.7,12.8
4,2,2024,1671,17.0,67.6,15.4
5,2,2025,830,18.7,64.6,16.7
6,3,2023,413,25.4,54.0,20.6
7,3,2024,1449,29.5,46.9,23.6
8,3,2025,1091,23.3,50.9,25.8
9,4,2023,117,14.5,70.9,14.5


In [ ]:

# STEP 8: YEAR-OVER-YEAR TRENDS
# Based on: Le Wagon Data Toolkit → Time series grouping


guest["year"] = guest["date_add"].dt.year
guest["month"] = guest["date_add"].dt.to_period("M")

# Sentiment by hotel + year
sentiment_trend = (
    guest.groupby(["hotel_id", "year"])["sentiment"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure columns exist
for col in ["NEG", "NEU", "POS"]:
    if col not in sentiment_trend.columns:
        sentiment_trend[col] = 0.0

# Add message counts
msg_counts = guest.groupby(["hotel_id", "year"]).size().reset_index(name="n_messages")
sentiment_trend = sentiment_trend.merge(msg_counts, on=["hotel_id", "year"])

# Format as percentages
sentiment_trend[["NEG", "NEU", "POS"]] = (sentiment_trend[["NEG", "NEU", "POS"]] * 100).round(1)

print("=== Sentiment % by Hotel + Year ===")
sentiment_trend[["hotel_id", "year", "n_messages", "NEG", "NEU", "POS"]]

=== Sentiment % by Hotel + Year ===


,hotel_id,year,n_messages,NEG,NEU,POS
0,1,2023,3509,20.5,62.5,17.0
1,1,2024,6172,21.6,64.6,13.7
2,1,2025,2524,20.6,64.1,15.3
3,2,2023,555,15.5,71.7,12.8
4,2,2024,1671,17.0,67.6,15.4
5,2,2025,830,18.7,64.6,16.7
6,3,2023,413,25.4,54.0,20.6
7,3,2024,1449,29.5,46.9,23.6
8,3,2025,1091,23.3,50.9,25.8
9,4,2023,117,14.5,70.9,14.5
